In [16]:
# Setup -- SQL right inside this notebook via sqlite3
import sqlite3
import pandas as pd
import os

conn = sqlite3.connect(':memory:')

def q(sql):
    return pd.read_sql_query(sql, conn)

def run(sql):
    conn.executescript(sql)
    conn.commit()


print('SQL engine ready.  sqlite3 version:', sqlite3.sqlite_version)


SQL engine ready.  sqlite3 version: 3.37.2


In [17]:
run('''
CREATE TABLE employees (
    employee_id INT PRIMARY KEY,
    employee_name VARCHAR(50),
    salary DECIMAL(10, 2),
    hire_date VARCHAR(20),
    department VARCHAR(50)
);

''')

In [18]:
run('''
-- Insert 20 sample records
INSERT INTO employees (employee_id, employee_name, salary, hire_date, department) VALUES
(1, 'Amy West', 60000.00, '2021-01-15', 'HR'),
(2, 'Ivy Lee', 75000.50, '2020-05-22', 'Sales'),
(3, 'joe smith', 80000.75, '2019-08-10', 'Marketing'),
(4, 'John White', 90000.00, '2020-11-05', 'Finance'),
(5, 'Jane Hill', 55000.25, '2022-02-28', 'IT'),
(6, 'Dave West', 72000.00, '2020-03-12', 'Marketing'),
(7, 'Fanny Lee', 85000.50, '2018-06-25', 'Sales'),
(8, 'Amy Smith', 95000.25, '2019-11-30', 'Finance'),
(9, 'Ivy Hill', 62000.75, '2021-07-18', 'IT'),
(10, 'Joe White', 78000.00, '2022-04-05', 'Marketing'),
(11, 'John Lee', 68000.50, '2018-12-10', 'HR'),
(12, 'Jane West', 89000.25, '2017-09-15', 'Sales'),
(13, 'Dave Smith', 60000.75, '2022-01-08', NULL),
(14, 'Fanny White', 72000.00, '2019-04-22', 'IT'),
(15, 'Amy Hill', 84000.50, '2020-08-17', 'Marketing'),
(16, 'Ivy West', 92000.25, '2021-02-03', 'Finance'),
(17, 'Joe Lee', 58000.75, '2018-05-28', 'IT'),
(18, 'John Smith', 77000.00, '2019-10-10', 'HR'),
(19, 'Jane Hill', 81000.50, '2022-03-15', 'Sales'),
(20, 'Dave White', 70000.25, '2017-12-20', 'Marketing');

''')

In [19]:
q('''

SELECT *
from employees
''')

,employee_id,employee_name,salary,hire_date,department
0,1,Amy West,60000.00,2021-01-15,HR
1,2,Ivy Lee,75000.50,2020-05-22,Sales
2,3,joe smith,80000.75,2019-08-10,Marketing
3,4,John White,90000.00,2020-11-05,Finance
4,5,Jane Hill,55000.25,2022-02-28,IT
5,6,Dave West,72000.00,2020-03-12,Marketing
6,7,Fanny Lee,85000.50,2018-06-25,Sales
7,8,Amy Smith,95000.25,2019-11-30,Finance
8,9,Ivy Hill,62000.75,2021-07-18,IT
9,10,Joe White,78000.00,2022-04-05,Marketing


In [ ]:
# Identify and handle any missing value.
# Check for and eliminate any duplicate rows in the dataset.
# Correct any structural issues, such as inconsistent naming conventions or formatting errors.
# Ensure all columns have appropriate data types (e.g. the hire_date column).
# Detect and address any outliers that may skew the analysis.
# Standardize and normalize data where applicable to ensure consistency.

In [20]:
# Identify and handle any missing value.

run('''
UPDATE employees
SET employee_id             = TRIM(employee_id ),
    employee_name = TRIM(employee_name),
    hire_date  = TRIM(hire_date),
    department    = TRIM(department),
    salary    = TRIM(salary )
''')



q('''
SELECT *
FROM employees
WHERE employee_id IS NULL
   OR LOWER(TRIM(employee_id)) IN ('null', 'unknown', 'n/a','none', '')

   OR employee_name IS NULL
   OR LOWER(TRIM(employee_name)) IN ('null', 'unknown', 'n/a', 'none','')

   OR hire_date IS NULL
   OR LOWER(TRIM(hire_date)) IN ('null', 'unknown', 'n/a','none', '')


   OR department IS NULL
   OR LOWER(TRIM(department)) IN ('null', 'unknown', 'n/a','none', '')

   OR salary IS NULL
   OR LOWER(TRIM(salary)) IN ('null', 'unknown', 'n/a','none', '')

''')

,employee_id,employee_name,salary,hire_date,department
0,13,Dave Smith,60000.75,2022-01-08,None


In [23]:
run('''
UPDATE employees
SET department = 'Unknown'
WHERE department IS NULL;
''')

In [25]:
q('''
SELECT *
FROM employees
group by employee_name ,salary ,hire_date ,department
having count(*)>1
''')



run('''
DELETE FROM employees
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM employees
    GROUP BY
         employee_name ,salary ,hire_date ,department
);
''')

In [27]:
run('''
UPDATE employees
SET employee_name = UPPER(SUBSTR(employee_name, 1, 1)) || LOWER(SUBSTR(employee_name, 2));
''')

In [29]:
q('''

SELECT
    MIN(salary) AS min_salary,
    MAX(salary) AS max_salary,
    AVG(salary) AS avg_salary
FROM employees;

''')

,min_salary,max_salary,avg_salary
0,55000.25,95000.25,75150.3375


In [30]:
q('''
SELECT DISTINCT department
FROM employees
ORDER BY department;

''')

,department
0,Finance
1,HR
2,IT
3,Marketing
4,Sales
5,Unknown


In [32]:
q('''
SELECT *
from employees
''')

,employee_id,employee_name,salary,hire_date,department
0,1,Amy west,60000.00,2021-01-15,HR
1,2,Ivy lee,75000.50,2020-05-22,Sales
2,3,Joe smith,80000.75,2019-08-10,Marketing
3,4,John white,90000.00,2020-11-05,Finance
4,5,Jane hill,55000.25,2022-02-28,IT
5,6,Dave west,72000.00,2020-03-12,Marketing
6,7,Fanny lee,85000.50,2018-06-25,Sales
7,8,Amy smith,95000.25,2019-11-30,Finance
8,9,Ivy hill,62000.75,2021-07-18,IT
9,10,Joe white,78000.00,2022-04-05,Marketing
